# LangGraph Support Ticket Router — Conditional Routing Deep Dive

## 1. Project Overview

**Task:** Build a LangGraph workflow that **classifies** incoming support tickets by difficulty, **routes** easy ones to automated resolution and hard ones to human review, and produces a structured output for each path.

**Why this matters:** Most support systems waste human time on tickets that could be auto-resolved (password resets, FAQ lookups, status checks). The key engineering challenge is building a **reliable router** — a conditional edge in the graph that inspects the ticket, decides the path, and sends it to the right handler without any manual triage.

**This notebook focuses on:**
- **Conditional routing** — the core LangGraph concept for branching workflows
- **Classification nodes** that produce structured decisions
- **Parallel resolution paths** with different logic per difficulty
- **Escalation mechanics** — when auto-resolution fails, re-route to humans

**Stack:**
- `LangGraph` — state graph orchestration with conditional edges
- `ChatOllama` + `qwen3.5:9b` — LLM (local, no API keys)
- `HuggingFaceEmbeddings` + `ChromaDB` — knowledge base for auto-resolution

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **conditional edges** — how LangGraph routes to different nodes based on state |
| 2 | Build a **classification node** that produces routing decisions with confidence scores |
| 3 | Implement **two parallel paths**: automated resolution vs human review |
| 4 | Add **escalation logic** — auto-resolution → human fallback when confidence is low |
| 5 | Design a **state schema** that supports branching and merging |
| 6 | Trace **which path each ticket took** and why |
| 7 | Evaluate routing accuracy against labeled ground truth |

## 3. Conditional Routing Explained

**Conditional routing** is the mechanism that makes LangGraph more powerful than a simple chain. Instead of A → B → C, you can have A → (if X then B, else C).

```
  ┌──────────────────────────────────────────────────────────┐
  │           HOW CONDITIONAL EDGES WORK                     │
  ├──────────────────────────────────────────────────────────┤
  │                                                          │
  │  1. A NODE runs and writes to state                      │
  │     e.g., state["difficulty"] = "easy"                   │
  │                                                          │
  │  2. A ROUTING FUNCTION inspects state                    │
  │     def route(state):                                    │
  │         if state["difficulty"] == "easy":                │
  │             return "auto_resolve"                        │
  │         else:                                            │
  │             return "human_review"                        │
  │                                                          │
  │  3. LangGraph follows the returned NODE NAME              │
  │     The routing function returns a string that matches    │
  │     a key in the edge mapping.                            │
  │                                                          │
  │  4. The EDGE MAPPING connects strings to nodes            │
  │     graph.add_conditional_edges(                          │
  │         source_node,                                     │
  │         routing_function,                                 │
  │         {"auto_resolve": "auto_node",                    │
  │          "human_review": "human_node"}                   │
  │     )                                                     │
  └──────────────────────────────────────────────────────────┘
```

### Key Rules

| Rule | Why |
|------|-----|
| Routing function **must be pure** — only reads state, no side effects | Predictable, testable routing |
| Return value **must match** a key in the edge mapping | Otherwise LangGraph raises an error |
| Routing happens **after** the source node completes | The source node must have already written the data the router needs |
| Multiple conditional edges are allowed | You can branch at multiple points in the graph |

## 4. Our Routing Architecture

```
  Ticket arrives
       │
       ▼
  ┌──────────────┐
  │  CLASSIFY     │  LLM classifies difficulty + category
  │  TICKET       │  → writes: difficulty, category, confidence
  └──────┬───────┘
         │
    CONDITIONAL EDGE ◄── route_by_difficulty(state)
    ┌────┴──────────┐
    ▼                ▼
  easy/medium       hard/escalation
    │                │
    ▼                ▼
  ┌──────────┐   ┌──────────────┐
  │ AUTO     │   │ HUMAN REVIEW │  Prepare human-readable
  │ RESOLVE  │   │ PACKAGE      │  summary, priority, context
  └────┬─────┘   └──────┬───────┘
       │                │
  CONDITIONAL           │
  ┌────┴────┐           │
  ▼         ▼           │
  high      low         │
  conf.     conf.       │
  │         │           │
  │      ESCALATE──────→│
  │                     │
  ▼                     ▼
  ┌──────────────────────┐
  │    FORMAT OUTPUT      │  Structured response per path
  └──────────┬───────────┘
             │
             ▼
           END
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers langgraph

print("Dependencies: langchain, langgraph, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports & Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import TypedDict, Literal

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, START, END
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"LangGraph: StateGraph, START, END")

## 7. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Knowledge Base for Auto-Resolution

## 8. FAQ / Runbook Knowledge Base

Easy tickets can be resolved automatically if we have a knowledge base of known solutions. This mimics a real support system's FAQ, runbooks, and standard operating procedures.

In [ ]:
KB_ARTICLES = [
    {"id": "KB001", "category": "account",
     "title": "Password Reset",
     "solution": "Go to login page → Click 'Forgot Password' → Enter email → Check inbox for reset link (valid 1 hour) → Create new password (min 12 chars, 1 uppercase, 1 number). If link expired, request a new one. If email not received, check spam or contact support with your account ID."},

    {"id": "KB002", "category": "account",
     "title": "Two-Factor Authentication Setup",
     "solution": "Settings → Security → Enable 2FA → Choose authenticator app or SMS → Scan QR code with your app → Enter the 6-digit code to verify → Save backup codes in a secure location. If locked out, use a backup code or contact support with photo ID for account recovery."},

    {"id": "KB003", "category": "billing",
     "title": "Update Payment Method",
     "solution": "Settings → Billing → Payment Methods → Add New → Enter card details → Set as default → Remove old card. Accepted: Visa, Mastercard, Amex. For invoicing, contact sales@company.com. Changes apply to next billing cycle."},

    {"id": "KB004", "category": "billing",
     "title": "Cancel Subscription",
     "solution": "Settings → Subscription → Cancel Plan → Select reason → Confirm. Access continues until period ends. Data retained 90 days after cancellation. Refund policy: full refund within 14 days of initial purchase, pro-rata after. Annual plans: contact billing team for partial refund."},

    {"id": "KB005", "category": "billing",
     "title": "Invoice and Receipt Download",
     "solution": "Settings → Billing → Billing History → Click invoice number → Download PDF. For tax purposes, receipts include company details and tax ID. Bulk download: select multiple invoices → Export. Historical invoices available for 7 years."},

    {"id": "KB006", "category": "technical",
     "title": "API Key Generation",
     "solution": "Dashboard → Developer → API Keys → Generate New Key → Name it → Set permissions (read/write/admin) → Copy immediately (shown once). Rate limits: 100 req/min (free), 2000 req/min (pro). Rotate keys every 90 days. Revoke compromised keys immediately."},

    {"id": "KB007", "category": "technical",
     "title": "Webhook Configuration",
     "solution": "Dashboard → Developer → Webhooks → Add Endpoint → Enter URL (HTTPS required) → Select events → Save. Test with 'Send Test Event'. Webhooks retry 3 times with exponential backoff. Verify signatures using the webhook secret. Timeout: 30 seconds."},

    {"id": "KB008", "category": "technical",
     "title": "Data Export",
     "solution": "Settings → Data → Export → Select data types (contacts, transactions, reports) → Choose format (CSV, JSON, XML) → Submit request. Large exports processed async — download link sent via email within 24 hours. Exports include data from last 5 years. GDPR/CCPA export requests handled within 2 business days."},

    {"id": "KB009", "category": "account",
     "title": "Change Email Address",
     "solution": "Settings → Profile → Email → Enter new email → Verify with current password → Confirmation sent to both old and new email. Old email receives a 48-hour undo link. SSO users: email must match your identity provider domain."},

    {"id": "KB010", "category": "account",
     "title": "Team Member Permissions",
     "solution": "Settings → Team → Members → Click member → Edit Role. Roles: Viewer (read-only), Editor (read/write), Admin (full access), Owner (billing + team management). Changes take effect immediately. Audit log tracks all permission changes."},
]

print(f"Knowledge base: {len(KB_ARTICLES)} articles")
for a in KB_ARTICLES:
    print(f"  [{a['id']}] {a['category']:<10} {a['title']}")

In [ ]:
# Index in ChromaDB
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("support_kb")
except Exception:
    pass

kb_collection = chroma_client.create_collection(
    name="support_kb", metadata={"hnsw:space": "cosine"}
)

kb_texts = [f"{a['title']}: {a['solution']}" for a in KB_ARTICLES]
kb_metas = [{"article_id": a["id"], "category": a["category"], "title": a["title"]} for a in KB_ARTICLES]
kb_ids = [a["id"] for a in KB_ARTICLES]
kb_vecs = embeddings.embed_documents(kb_texts)
kb_collection.add(documents=kb_texts, embeddings=kb_vecs, metadatas=kb_metas, ids=kb_ids)

print(f"Indexed {kb_collection.count()} KB articles")

## 9. Test Tickets

A labeled set of support tickets with known difficulty and expected routing.

In [ ]:
TEST_TICKETS = [
    # Easy — direct KB matches
    {"id": "T01", "text": "I forgot my password and can't log in. How do I reset it?",
     "expected_route": "auto", "expected_category": "account", "expected_difficulty": "easy"},

    {"id": "T02", "text": "How do I change my credit card on file? My old card expired.",
     "expected_route": "auto", "expected_category": "billing", "expected_difficulty": "easy"},

    {"id": "T03", "text": "I need to download my invoices from last year for tax filing.",
     "expected_route": "auto", "expected_category": "billing", "expected_difficulty": "easy"},

    {"id": "T04", "text": "How do I generate a new API key? I need write permissions.",
     "expected_route": "auto", "expected_category": "technical", "expected_difficulty": "easy"},

    {"id": "T05", "text": "I want to cancel my subscription. What happens to my data?",
     "expected_route": "auto", "expected_category": "billing", "expected_difficulty": "easy"},

    # Medium — partially covered by KB, may need human judgment
    {"id": "T06", "text": "I was charged twice for my subscription this month. One charge on the 1st and another on the 15th. I need a refund for the duplicate.",
     "expected_route": "auto", "expected_category": "billing", "expected_difficulty": "medium"},

    {"id": "T07", "text": "My webhooks keep failing with 500 errors. I've checked my endpoint and it works when I curl it directly. Started happening yesterday.",
     "expected_route": "human", "expected_category": "technical", "expected_difficulty": "medium"},

    # Hard — require human expertise, investigation, or judgment
    {"id": "T08", "text": "We're seeing a significant data discrepancy between our reports in your dashboard and our internal analytics. Transaction counts are about 15% lower in your system for the past 2 weeks. This is affecting our financial reconciliation.",
     "expected_route": "human", "expected_category": "technical", "expected_difficulty": "hard"},

    {"id": "T09", "text": "Our company was recently acquired. We need to transfer the account ownership, merge billing with the parent company, and migrate our data to their existing enterprise tenant. There are also compliance concerns about data residency since the parent is EU-based.",
     "expected_route": "human", "expected_category": "account", "expected_difficulty": "hard"},

    {"id": "T10", "text": "I believe there is a security vulnerability in your API. When I send a specially crafted request to the /users endpoint, I can see other users' email addresses in the response headers. I have not exploited this but wanted to report it responsibly.",
     "expected_route": "human", "expected_category": "security", "expected_difficulty": "hard"},

    {"id": "T11", "text": "We are under contractual obligation to have less than 2 hours of downtime per quarter. Your service was down for 3 hours last Tuesday and 45 minutes on Thursday. I need an official incident report and SLA credit calculation.",
     "expected_route": "human", "expected_category": "technical", "expected_difficulty": "hard"},

    {"id": "T12", "text": "I accidentally deleted our production webhook endpoint. Can you restore it with the same secret? We have about 500 events queued that haven't been delivered.",
     "expected_route": "human", "expected_category": "technical", "expected_difficulty": "medium"},
]

from collections import Counter
print(f"Test tickets: {len(TEST_TICKETS)}")
print(f"Expected routes: {dict(Counter(t['expected_route'] for t in TEST_TICKETS))}")
print(f"Expected difficulty: {dict(Counter(t['expected_difficulty'] for t in TEST_TICKETS))}")

---

# Part B — State Schema & Graph Nodes

## 10. State Schema

In [ ]:
class TicketState(TypedDict):
    """State that flows through the support ticket routing graph.

    Written by different nodes:
      classify_ticket  → difficulty, category, confidence, classification_reasoning
      auto_resolve     → kb_matches, auto_response, auto_confidence
      human_review     → human_summary, priority, escalation_reason
      format_output    → final_output
    """
    # Input
    ticket_id: str
    ticket_text: str
    # Classification (set by classify_ticket)
    difficulty: str            # "easy", "medium", "hard"
    category: str              # "account", "billing", "technical", "security"
    confidence: float          # 0.0 to 1.0
    classification_reasoning: str
    # Auto-resolution (set by auto_resolve)
    kb_matches: list[dict]
    auto_response: str
    auto_confidence: float     # confidence that auto-resolution is correct
    # Human review (set by human_review)
    human_summary: str
    priority: str              # "P1", "P2", "P3", "P4"
    escalation_reason: str
    # Output (set by format_output)
    route_taken: str           # "auto", "human", "auto→human"
    final_output: str
    current_step: str


print("State schema: TicketState")
print(f"Fields: {len(TicketState.__annotations__)}")
for name in TicketState.__annotations__:
    print(f"  {name}")

## 11. Node 1: Classify Ticket

The classifier determines **difficulty** and **category**. The difficulty drives the conditional routing decision.

Classification criteria:
- **Easy**: Standard questions with known answers — password resets, how-to, settings changes
- **Medium**: Clear questions but may need investigation or partial human involvement
- **Hard**: Complex situations, security issues, compliance, multi-step business operations

In [ ]:
CLASSIFY_SYSTEM = """You are a support ticket classifier. Analyze the ticket and classify it.

Difficulty levels:
- easy: Standard questions with well-known solutions (password reset, settings, how-to)
- medium: Needs some investigation but likely resolvable with standard procedures
- hard: Complex issues requiring human judgment, multi-department coordination, security, or compliance

Categories: account, billing, technical, security

Confidence: How certain are you about the classification (0.0 to 1.0)."""

CLASSIFY_PROMPT = """SUPPORT TICKET:
{ticket_text}

Classify this ticket. Return ONLY JSON:
{{
  "difficulty": "easy" or "medium" or "hard",
  "category": "account" or "billing" or "technical" or "security",
  "confidence": 0.0 to 1.0,
  "reasoning": "brief explanation of why this difficulty level"
}}"""


def classify_ticket(state: TicketState) -> dict:
    """Node: Classify ticket difficulty and category."""
    print(f"  [NODE] classify_ticket — {state['ticket_id']}")

    result = parse_json(ask(
        CLASSIFY_PROMPT.format(ticket_text=state["ticket_text"]),
        system=CLASSIFY_SYSTEM, temperature=0.0,
    ))

    if result:
        difficulty = result.get("difficulty", "hard")
        category = result.get("category", "technical")
        confidence = float(result.get("confidence", 0.5))
        reasoning = result.get("reasoning", "")
    else:
        # Fallback: if classification fails, route to human (safe default)
        difficulty = "hard"
        category = "technical"
        confidence = 0.0
        reasoning = "Classification failed — defaulting to human review"

    print(f"    → difficulty={difficulty}, category={category}, confidence={confidence:.2f}")

    return {
        "difficulty": difficulty,
        "category": category,
        "confidence": confidence,
        "classification_reasoning": reasoning,
        "current_step": "classify_ticket",
    }


print("Node defined: classify_ticket")

## 12. Routing Function — The Core Conditional Edge

This is the most important function. It reads the classification state and decides which path to take.

```
  classify_ticket
       │
  route_by_difficulty(state)
       │
  ┌────┴──────────────┐
  │                    │
  ▼                    ▼
  easy/medium          hard OR low confidence
  + high confidence
  │                    │
  ▼                    ▼
  auto_resolve         human_review
```

In [ ]:
CONFIDENCE_THRESHOLD = 0.6  # Below this, route to human regardless of difficulty


def route_by_difficulty(state: TicketState) -> str:
    """Conditional edge: route ticket based on difficulty and confidence.

    This function is PURE — it only reads state and returns a string.
    It does not modify state or call external services.

    Routing logic:
      1. Hard difficulty → always human
      2. Low confidence (any difficulty) → human (we're not sure enough)
      3. Easy/medium + high confidence → auto-resolve
    """
    difficulty = state.get("difficulty", "hard")
    confidence = state.get("confidence", 0.0)

    if difficulty == "hard":
        print(f"    [ROUTE] hard → human_review")
        return "human_review"

    if confidence < CONFIDENCE_THRESHOLD:
        print(f"    [ROUTE] low confidence ({confidence:.2f}) → human_review")
        return "human_review"

    print(f"    [ROUTE] {difficulty} + conf={confidence:.2f} → auto_resolve")
    return "auto_resolve"


print(f"Routing function defined: route_by_difficulty")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"  easy/medium + conf >= {CONFIDENCE_THRESHOLD} → auto_resolve")
print(f"  hard OR conf < {CONFIDENCE_THRESHOLD} → human_review")

## 13. Node 2a: Auto-Resolve (Easy Path)

For easy tickets, search the knowledge base and generate an automated response.

In [ ]:
RESOLVE_SYSTEM = """You are a support assistant. Answer the customer's question using ONLY
the provided knowledge base articles. Be friendly, specific, and include
step-by-step instructions when applicable. If the KB doesn't cover the issue,
say so clearly."""

RESOLVE_PROMPT = """CUSTOMER TICKET:
{ticket_text}

KNOWLEDGE BASE ARTICLES:
{kb_context}

Write a helpful response. Also assess your confidence that this fully resolves
the customer's issue (0.0 to 1.0).

Return JSON:
{{
  "response": "your helpful response to the customer",
  "confidence": 0.0 to 1.0,
  "matches_kb": true or false
}}"""


def auto_resolve(state: TicketState) -> dict:
    """Node: Attempt to resolve ticket automatically using KB."""
    print(f"  [NODE] auto_resolve — {state['ticket_id']}")

    # Search KB
    qv = embeddings.embed_query(state["ticket_text"])
    results = kb_collection.query(query_embeddings=[qv], n_results=3)

    kb_matches = []
    for i in range(len(results["documents"][0])):
        kb_matches.append({
            "article_id": results["metadatas"][0][i]["article_id"],
            "title": results["metadatas"][0][i]["title"],
            "text": results["documents"][0][i],
            "similarity": round(1 - results["distances"][0][i], 4),
        })
    print(f"    KB matches: {[m['article_id'] for m in kb_matches]} "
          f"(scores: {[m['similarity'] for m in kb_matches]})")

    # Generate response
    kb_context = "\n\n".join(
        f"[{m['article_id']}] {m['text']}" for m in kb_matches
    )

    result = parse_json(ask(
        RESOLVE_PROMPT.format(ticket_text=state["ticket_text"], kb_context=kb_context),
        system=RESOLVE_SYSTEM, temperature=0.1,
    ))

    if result:
        auto_response = result.get("response", "")
        auto_confidence = float(result.get("confidence", 0.5))
    else:
        auto_response = "I'm sorry, I wasn't able to generate a response. Let me transfer you to a human agent."
        auto_confidence = 0.0

    print(f"    Auto-resolve confidence: {auto_confidence:.2f}")

    return {
        "kb_matches": kb_matches,
        "auto_response": auto_response,
        "auto_confidence": auto_confidence,
        "current_step": "auto_resolve",
    }


print("Node defined: auto_resolve")

## 14. Second Conditional Edge — Escalation Check

After auto-resolution, we check if the auto-generated answer is confident enough. If not, **escalate to human review**. This is the second conditional routing point.

```
  auto_resolve
       │
  check_auto_confidence(state)
       │
  ┌────┴──────────┐
  ▼                ▼
  confident        not confident
  │                │
  ▼                ▼
  format_output    escalate_to_human
```

In [ ]:
AUTO_CONFIDENCE_THRESHOLD = 0.7


def check_auto_confidence(state: TicketState) -> str:
    """Conditional edge: after auto-resolve, check if we should escalate.

    If the auto-response confidence is too low, route to human review
    instead of sending a potentially wrong answer to the customer.
    """
    auto_conf = state.get("auto_confidence", 0.0)

    if auto_conf >= AUTO_CONFIDENCE_THRESHOLD:
        print(f"    [ROUTE] auto confidence {auto_conf:.2f} >= {AUTO_CONFIDENCE_THRESHOLD} → format_output")
        return "format_output"
    else:
        print(f"    [ROUTE] auto confidence {auto_conf:.2f} < {AUTO_CONFIDENCE_THRESHOLD} → escalate")
        return "escalate_to_human"


print(f"Second routing function: check_auto_confidence")
print(f"  Auto confidence >= {AUTO_CONFIDENCE_THRESHOLD} → format_output (send response)")
print(f"  Auto confidence <  {AUTO_CONFIDENCE_THRESHOLD} → escalate_to_human (fallback)")

## 15. Node 2b: Human Review (Hard Path)

In [ ]:
TRIAGE_SYSTEM = """You are a support escalation specialist. Prepare a clear handoff package
for a human agent. Summarize the issue, assess priority, and suggest first steps."""

TRIAGE_PROMPT = """SUPPORT TICKET [{ticket_id}]:
{ticket_text}

CLASSIFICATION:
Difficulty: {difficulty}
Category: {category}
Reason: {reasoning}

{escalation_context}

Prepare a human review package. Return JSON:
{{
  "summary": "1-2 sentence summary for the agent",
  "priority": "P1" (urgent, customer impact) or "P2" (high, business impact) or "P3" (normal) or "P4" (low),
  "suggested_actions": ["first step", "second step"],
  "skills_needed": ["billing", "engineering", "security", "legal", etc.]
}}"""


def human_review(state: TicketState) -> dict:
    """Node: Prepare ticket for human agent review."""
    print(f"  [NODE] human_review — {state['ticket_id']}")

    escalation_context = ""
    if state.get("auto_response"):
        escalation_context = (
            f"ESCALATION NOTE: Auto-resolution was attempted but confidence was low "
            f"({state.get('auto_confidence', 0):.2f}). Auto-response draft:\n"
            f"{state['auto_response'][:200]}"
        )

    result = parse_json(ask(
        TRIAGE_PROMPT.format(
            ticket_id=state["ticket_id"],
            ticket_text=state["ticket_text"],
            difficulty=state.get("difficulty", "?"),
            category=state.get("category", "?"),
            reasoning=state.get("classification_reasoning", ""),
            escalation_context=escalation_context,
        ),
        system=TRIAGE_SYSTEM, temperature=0.1,
    ))

    if result:
        summary = result.get("summary", "")
        priority = result.get("priority", "P3")
        actions = result.get("suggested_actions", [])
        skills = result.get("skills_needed", [])
    else:
        summary = f"Ticket {state['ticket_id']} requires human review."
        priority = "P3"
        actions = ["Review ticket details"]
        skills = [state.get("category", "general")]

    escalation_reason = "Direct route (hard ticket)" if not state.get("auto_response") else "Escalated from auto-resolution (low confidence)"

    print(f"    Priority: {priority}, Skills: {skills}")

    return {
        "human_summary": summary,
        "priority": priority,
        "escalation_reason": escalation_reason,
        "route_taken": "auto→human" if state.get("auto_response") else "human",
        "current_step": "human_review",
    }


print("Node defined: human_review")

## 16. Escalation Node

In [ ]:
def escalate_to_human(state: TicketState) -> dict:
    """Node: Transition from auto-resolve failure to human review.

    This is a thin passthrough that marks the route as escalated.
    The actual human review prep happens in the human_review node.
    """
    print(f"  [NODE] escalate_to_human — {state['ticket_id']}")
    print(f"    Auto-resolution failed (confidence={state.get('auto_confidence', 0):.2f})")
    print(f"    Escalating to human review...")

    return {
        "escalation_reason": f"Auto-resolution confidence too low ({state.get('auto_confidence', 0):.2f})",
        "current_step": "escalate_to_human",
    }


print("Node defined: escalate_to_human")

## 17. Node 3: Format Output

Both paths converge here to produce a structured final output.

In [ ]:
def format_output(state: TicketState) -> dict:
    """Node: Produce the final structured output for this ticket."""
    print(f"  [NODE] format_output — {state['ticket_id']}")

    route = state.get("route_taken", "")
    if not route:
        route = "auto" if state.get("auto_response") and not state.get("human_summary") else "human"

    lines = []
    lines.append(f"=== TICKET {state['ticket_id']} — RESOLUTION ===")
    lines.append(f"Route: {route.upper()}")
    lines.append(f"Category: {state.get('category', '?')}")
    lines.append(f"Difficulty: {state.get('difficulty', '?')}")
    lines.append(f"Classification confidence: {state.get('confidence', 0):.0%}")
    lines.append("")

    if route == "auto":
        lines.append("--- AUTOMATED RESPONSE ---")
        lines.append(state.get("auto_response", "(no response)"))
        lines.append(f"\nResolution confidence: {state.get('auto_confidence', 0):.0%}")
        if state.get("kb_matches"):
            articles = [m["article_id"] for m in state["kb_matches"][:2]]
            lines.append(f"KB articles used: {articles}")
    else:
        lines.append("--- ROUTED TO HUMAN AGENT ---")
        lines.append(f"Priority: {state.get('priority', '?')}")
        lines.append(f"Reason: {state.get('escalation_reason', '?')}")
        lines.append(f"Summary: {state.get('human_summary', '(none)')}")
        if state.get("auto_response"):
            lines.append(f"\nDraft auto-response (for agent reference):")
            lines.append(state["auto_response"][:200])

    final_output = "\n".join(lines)
    print(f"    Output: {len(final_output)} chars, route={route}")

    return {
        "route_taken": route,
        "final_output": final_output,
        "current_step": "format_output",
    }


print("Node defined: format_output")

---

# Part C — Assemble & Compile the Graph

## 18. Wire the Graph

This graph has **two conditional edges** — the primary routing decision after classification and the escalation check after auto-resolution.

In [ ]:
# Build the graph
workflow = StateGraph(TicketState)

# --- Add nodes ---
workflow.add_node("classify_ticket", classify_ticket)
workflow.add_node("auto_resolve", auto_resolve)
workflow.add_node("escalate_to_human", escalate_to_human)
workflow.add_node("human_review", human_review)
workflow.add_node("format_output", format_output)

# --- Add edges ---
#  START → classify_ticket (always)
workflow.add_edge(START, "classify_ticket")

#  CONDITIONAL EDGE 1: classify → route by difficulty
workflow.add_conditional_edges(
    "classify_ticket",
    route_by_difficulty,
    {
        "auto_resolve": "auto_resolve",
        "human_review": "human_review",
    },
)

#  CONDITIONAL EDGE 2: auto_resolve → check confidence
workflow.add_conditional_edges(
    "auto_resolve",
    check_auto_confidence,
    {
        "format_output": "format_output",
        "escalate_to_human": "escalate_to_human",
    },
)

#  escalate → human_review
workflow.add_edge("escalate_to_human", "human_review")

#  human_review → format_output
workflow.add_edge("human_review", "format_output")

#  format_output → END
workflow.add_edge("format_output", END)

# Compile
app = workflow.compile()

print("Graph compiled!")
print()
print("GRAPH STRUCTURE:")
print("  START → classify_ticket")
print("  classify_ticket ─(easy/medium + high conf)─→ auto_resolve")
print("  classify_ticket ─(hard OR low conf)────────→ human_review")
print("  auto_resolve ───(high auto conf)───────────→ format_output")
print("  auto_resolve ───(low auto conf)────────────→ escalate_to_human → human_review")
print("  human_review ─────────────────────────────→ format_output")
print("  format_output ────────────────────────────→ END")
print()
print("Two conditional edges:")
print(f"  1. route_by_difficulty (threshold: {CONFIDENCE_THRESHOLD})")
print(f"  2. check_auto_confidence (threshold: {AUTO_CONFIDENCE_THRESHOLD})")

In [ ]:
# Visualize graph structure
try:
    mermaid_str = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid_str)
except Exception as e:
    print(f"Mermaid rendering: {e}")
    print("\nGraph nodes and edges listed above.")

---

# Part D — Run All Tickets Through the Graph

## 19. Execute Batch Processing

In [ ]:
def run_ticket(ticket: dict) -> dict:
    """Run a single ticket through the routing graph."""
    initial_state = {
        "ticket_id": ticket["id"],
        "ticket_text": ticket["text"],
        "difficulty": "",
        "category": "",
        "confidence": 0.0,
        "classification_reasoning": "",
        "kb_matches": [],
        "auto_response": "",
        "auto_confidence": 0.0,
        "human_summary": "",
        "priority": "",
        "escalation_reason": "",
        "route_taken": "",
        "final_output": "",
        "current_step": "start",
    }

    result = app.invoke(initial_state, {"recursion_limit": 10})
    return result


print("PROCESSING ALL TICKETS")
print("=" * 70)

all_results = []
for i, ticket in enumerate(TEST_TICKETS, 1):
    print(f"\n{'─' * 70}")
    print(f"[{i}/{len(TEST_TICKETS)}] Ticket {ticket['id']}: {ticket['text'][:60]}...")
    print(f"  Expected: route={ticket['expected_route']}, "
          f"difficulty={ticket['expected_difficulty']}, category={ticket['expected_category']}")

    result = run_ticket(ticket)
    all_results.append({"ticket": ticket, "result": result})

    actual_route = result.get("route_taken", "?")
    print(f"  Actual:   route={actual_route}, "
          f"difficulty={result.get('difficulty', '?')}, category={result.get('category', '?')}")

print(f"\n{'=' * 70}")
print(f"All {len(all_results)} tickets processed.")

---

# Part E — Evaluate Routing Quality

## 20. Routing Accuracy

In [ ]:
print("ROUTING ACCURACY")
print("=" * 95)
print(f"{'TID':<5} {'Expected':<10} {'Actual':<12} {'Match':<6} {'Diff':<8} {'Cat':<10} {'Conf':<6} {'Steps'}")
print("-" * 95)

route_correct = 0
diff_correct = 0
cat_correct = 0

for item in all_results:
    t = item["ticket"]
    r = item["result"]

    actual_route = r.get("route_taken", "?")
    # Normalize: "auto→human" counts as "human" for routing accuracy
    actual_route_norm = "human" if "human" in actual_route else "auto"
    expected_route = t["expected_route"]

    r_match = actual_route_norm == expected_route
    d_match = r.get("difficulty", "?") == t["expected_difficulty"]
    c_match = r.get("category", "?") == t["expected_category"]

    route_correct += int(r_match)
    diff_correct += int(d_match)
    cat_correct += int(c_match)

    route_icon = "+" if r_match else "-"
    steps = r.get("current_step", "?")

    print(f"{t['id']:<5} {expected_route:<10} {actual_route:<12} [{route_icon}]   "
          f"{r.get('difficulty', '?'):<8} {r.get('category', '?'):<10} "
          f"{r.get('confidence', 0):<6.0%} {steps}")

n = len(all_results)
print(f"\n  ACCURACY:")
print(f"    Routing:       {route_correct}/{n} ({route_correct/n:.0%})")
print(f"    Difficulty:    {diff_correct}/{n} ({diff_correct/n:.0%})")
print(f"    Category:      {cat_correct}/{n} ({cat_correct/n:.0%})")

In [ ]:
# Routing confusion matrix
print("ROUTING CONFUSION MATRIX")
print("=" * 40)
print(f"{'':>15} {'Predicted':>10}")
print(f"{'':>15} {'Auto':>5} {'Human':>6}")
print("-" * 28)

tp = fp = fn = tn = 0
for item in all_results:
    expected = item["ticket"]["expected_route"]
    actual = item["result"].get("route_taken", "?")
    actual_norm = "human" if "human" in actual else "auto"

    if expected == "auto" and actual_norm == "auto":
        tp += 1
    elif expected == "auto" and actual_norm == "human":
        fn += 1  # should have been auto, was sent to human (safe fail)
    elif expected == "human" and actual_norm == "auto":
        fp += 1  # should have been human, was auto-resolved (DANGEROUS)
    else:
        tn += 1

print(f"  Actual Auto  {tp:>5} {fn:>6}")
print(f"  Actual Human {fp:>5} {tn:>6}")
print()
print(f"  True Positives (correct auto):     {tp}")
print(f"  True Negatives (correct human):     {tn}")
print(f"  False Negatives (over-escalated):   {fn} — safe but wasteful")
print(f"  False Positives (under-escalated):  {fp} — DANGEROUS, wrong auto-response")

if fp > 0:
    print(f"\n  WARNING: {fp} tickets were auto-resolved but should have gone to a human!")
    for item in all_results:
        if item["ticket"]["expected_route"] == "human" and "human" not in item["result"].get("route_taken", ""):
            print(f"    {item['ticket']['id']}: {item['ticket']['text'][:60]}...")
else:
    print(f"\n  No dangerous mis-routes detected.")

## 21. Path Distribution

In [ ]:
# How many tickets took each path?
print("ROUTING PATH DISTRIBUTION")
print("=" * 50)

path_counts = {}
for item in all_results:
    route = item["result"].get("route_taken", "unknown")
    path_counts[route] = path_counts.get(route, 0) + 1

total = len(all_results)
for path, count in sorted(path_counts.items()):
    bar = "█" * (count * 3)
    print(f"  {path:<12} {count:>2} ({count/total:.0%}) {bar}")

auto_count = sum(1 for item in all_results
                 if item["result"].get("route_taken") == "auto")
human_count = total - auto_count

print(f"\n  Auto-resolved: {auto_count}/{total} ({auto_count / total:.0%})")
print(f"  Human review:  {human_count}/{total} ({human_count / total:.0%})")
print(f"\n  If agents handle 1 ticket/10min, auto-resolution saves ~{auto_count * 10} min/batch.")

## 22. Sample Outputs by Path

In [ ]:
# Show one auto-resolved and one human-reviewed output
auto_example = next((item for item in all_results
                     if item["result"].get("route_taken") == "auto"), None)
human_example = next((item for item in all_results
                      if item["result"].get("route_taken") == "human"), None)

if auto_example:
    print("SAMPLE: AUTO-RESOLVED TICKET")
    print("=" * 70)
    print(f"Ticket: {auto_example['ticket']['text'][:80]}")
    print()
    wrap_print(auto_example["result"]["final_output"])

print()

if human_example:
    print("SAMPLE: HUMAN-ROUTED TICKET")
    print("=" * 70)
    print(f"Ticket: {human_example['ticket']['text'][:80]}")
    print()
    wrap_print(human_example["result"]["final_output"])

In [ ]:
# Show an escalated ticket if any
escalated = next((item for item in all_results
                  if item["result"].get("route_taken") == "auto→human"), None)

if escalated:
    print("SAMPLE: ESCALATED TICKET (auto → human)")
    print("=" * 70)
    print(f"Ticket: {escalated['ticket']['text'][:80]}")
    print(f"Auto confidence: {escalated['result'].get('auto_confidence', 0):.0%}")
    print(f"Escalation reason: {escalated['result'].get('escalation_reason', '?')}")
    print()
    wrap_print(escalated["result"]["final_output"])
else:
    print("No tickets were escalated from auto to human.")
    print("This means all auto-resolved tickets had high enough confidence.")
    print(f"(Threshold: {AUTO_CONFIDENCE_THRESHOLD})")

---

# Part F — Deep Dive: Conditional Routing Patterns

## 23. Routing Pattern Comparison

Let's explain the three common conditional routing patterns and when to use each.

In [ ]:
print("CONDITIONAL ROUTING PATTERNS")
print("=" * 80)

print("""
PATTERN 1: BINARY SPLIT
The simplest conditional — one criterion, two paths.

    classify → easy? ──yes──→ auto_resolve
                   └──no───→ human_review

    def route(state):
        if state["difficulty"] == "easy":
            return "auto_resolve"
        return "human_review"

    When to use: Two clear categories, no nuance needed.
""")

print("""
PATTERN 2: MULTI-WAY SPLIT (what we use)
Multiple criteria combined to produce routing decisions.

    classify → difficulty + confidence ──→ auto_resolve
                                    ├───→ human_review
                                    └───→ (escalate fallback)

    def route(state):
        if state["difficulty"] == "hard":
            return "human_review"
        if state["confidence"] < threshold:
            return "human_review"
        return "auto_resolve"

    When to use: Nuanced decisions with confidence-based fallbacks.
""")

print("""
PATTERN 3: CASCADING CONDITIONALS
Multiple conditional edges in sequence — decision at multiple points.

    classify ──→ auto_resolve ──→ confidence check ──→ output
         │                                      └───→ human_review
         └──→ human_review ────────────────────────→ output

    This is the most powerful pattern: try the fast path first,
    then fall back to the slow path if the fast path isn't confident.

    Our graph uses BOTH Pattern 2 (at classify) and Pattern 3 (at auto_resolve).
""")

## 24. Confidence Threshold Sensitivity

In [ ]:
# What happens if we change the confidence threshold?
print("THRESHOLD SENSITIVITY ANALYSIS")
print("=" * 70)
print("How routing accuracy and auto-resolve rate change with threshold:\n")

thresholds = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]

print(f"{'Threshold':<12} {'Auto-resolved':<16} {'Correct route':<16} {'Dangerous mis-routes'}")
print("-" * 65)

for threshold in thresholds:
    auto = 0
    correct = 0
    dangerous = 0

    for item in all_results:
        t = item["ticket"]
        r = item["result"]

        difficulty = r.get("difficulty", "hard")
        confidence = r.get("confidence", 0.0)

        # Simulate routing with this threshold
        if difficulty == "hard" or confidence < threshold:
            sim_route = "human"
        else:
            sim_route = "auto"

        if sim_route == "auto":
            auto += 1

        expected = t["expected_route"]
        if sim_route == expected:
            correct += 1
        elif expected == "human" and sim_route == "auto":
            dangerous += 1

    n = len(all_results)
    marker = " ◄── current" if threshold == CONFIDENCE_THRESHOLD else ""
    print(f"  {threshold:<12} {auto:>2}/{n} ({auto/n:>4.0%})       "
          f"{correct:>2}/{n} ({correct/n:>4.0%})       {dangerous}{marker}")

print(f"\n  Higher threshold → fewer auto-resolves but fewer dangerous errors.")
print(f"  Lower threshold → more auto-resolves but risk of wrong auto-responses.")
print(f"  Optimal threshold depends on cost of human time vs cost of wrong answers.")

## 25. Understanding State After Each Node

In [ ]:
# Trace a single ticket step by step to show exact state changes
trace_ticket = TEST_TICKETS[0]  # Easy password reset ticket

print(f"STEP-BY-STEP STATE TRACE: {trace_ticket['id']}")
print(f"Ticket: {trace_ticket['text']}")
print("=" * 80)

trace_state = {
    "ticket_id": trace_ticket["id"],
    "ticket_text": trace_ticket["text"],
    "difficulty": "", "category": "", "confidence": 0.0,
    "classification_reasoning": "",
    "kb_matches": [], "auto_response": "", "auto_confidence": 0.0,
    "human_summary": "", "priority": "", "escalation_reason": "",
    "route_taken": "", "final_output": "", "current_step": "start",
}

step_num = 0
for step_output in app.stream(trace_state, {"recursion_limit": 10}):
    step_num += 1
    for node_name, partial_update in step_output.items():
        print(f"\n  Step {step_num}: {node_name}")
        print(f"  State changes (partial update returned by this node):")
        for key, value in partial_update.items():
            if isinstance(value, str) and len(value) > 80:
                print(f"    {key} = '{value[:80]}...'")
            elif isinstance(value, list) and value:
                print(f"    {key} = [{len(value)} items]")
            else:
                print(f"    {key} = {value}")

print(f"\n  Total steps: {step_num}")
print(f"  Each step is a node that reads state, does work, returns a PARTIAL update.")
print(f"  LangGraph merges each partial update into the accumulated state.")

---

## 26. Common Mistakes in Conditional Routing

| Mistake | Why It's Wrong | Fix |
|---------|---------------|-----|
| Routing function has side effects | Makes debugging impossible, routing becomes unpredictable | Routing functions must ONLY read state and return a string |
| Missing edge mapping key | If routing function returns `"foo"` but `"foo"` isn't in the mapping, graph crashes | Ensure every possible return value has a mapping entry |
| No fallback for low confidence | Silently sends bad auto-responses to customers | Always have an escalation path when confidence is below threshold |
| Single routing point | Can't catch errors in auto-resolution | Use cascading conditionals — route at classification AND after resolution |
| Hardcoded thresholds | One threshold doesn't fit all categories | Consider per-category thresholds (billing may need higher confidence than FAQ) |
| Not testing all paths | Escalation path never tested, breaks in production | Run test tickets that exercise every possible route |

## 27. Production Considerations

1. **Logging**: Record every routing decision with the state snapshot — essential for debugging and tuning thresholds
2. **A/B testing**: Run shadow mode where hard tickets get auto-responses that aren't sent, but are scored against human responses
3. **Feedback loop**: Track customer satisfaction per route (CSAT) to tune confidence thresholds
4. **Priority queuing**: Human-routed tickets should be sorted by priority (P1 before P4)
5. **SLA tracking**: Monitor time-to-response for each path — auto-resolution should be <1 second, human should meet SLA targets
6. **Batch routing**: For high-volume systems, classify in batch and then route — reduces LLM calls

## 28. Exercises

### Exercise 1
Add a **sentiment analysis node** between classification and routing. If the customer sounds angry (high negative sentiment), escalate to a senior agent regardless of difficulty. Modify the routing function to check `state["sentiment"]`.

### Exercise 2
Implement **per-category confidence thresholds**: billing tickets should require 80% confidence for auto-resolution (financial impact), but FAQ/account tickets only need 60%. Update `route_by_difficulty` to check category-specific thresholds.

### Exercise 3
Build a **feedback node** after `format_output`: simulate the customer responding with "this didn't help" or "thanks, that worked". If negative feedback, escalate to human. This adds a third conditional edge at the end of the graph.

### Mini Challenge
Implement a **multi-tier routing graph**: Tier 1 (KB auto-response), Tier 2 (senior agent with context), Tier 3 (specialist with full investigation). Each tier has its own resolution attempt and confidence check, with cascading escalation. Track the cost savings from resolving tickets at lower tiers.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Tickets processed:     {len(all_results)}")
print(f"KB articles:           {len(KB_ARTICLES)}")
print()
print(f"ROUTING RESULTS:")
for path, count in sorted(path_counts.items()):
    print(f"  {path:<14} {count} tickets")
print(f"  Routing accuracy:  {route_correct}/{n} ({route_correct/n:.0%})")
print(f"  Dangerous errors:  {fp}")
print()
print(f"GRAPH STRUCTURE:")
print(f"  Nodes:               5 (classify, auto_resolve, escalate, human_review, format)")
print(f"  Conditional edges:   2 (route_by_difficulty, check_auto_confidence)")
print(f"  Possible paths:      3 (auto, human, auto→human escalation)")
print()
print(f"THRESHOLDS:")
print(f"  Classification:      {CONFIDENCE_THRESHOLD} (below → human)")
print(f"  Auto-resolution:     {AUTO_CONFIDENCE_THRESHOLD} (below → escalate)")
print()
print(f"Components built:")
print(f"  1. Ticket classifier      — difficulty + category + confidence")
print(f"  2. Primary router         — conditional edge on difficulty/confidence")
print(f"  3. KB-backed auto-resolve — vector search + LLM response generation")
print(f"  4. Escalation checker     — second conditional edge on auto confidence")
print(f"  5. Human review packager  — priority assignment + agent summary")
print(f"  6. Output formatter       — unified output for all paths")
print(f"  7. Routing evaluator      — accuracy, confusion matrix, threshold analysis")

## 29. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Conditional edges** are the core power of LangGraph — they let you branch workflows based on computed state |
| 2 | **Routing functions must be pure** — only read state, return a string matching the edge mapping |
| 3 | **Two-level routing** (classify then verify) catches more errors than a single routing decision |
| 4 | **Cascading conditionals** let you try the fast path first, then fall back to the slow path |
| 5 | **Confidence thresholds** should err on the side of escalation — a wrong auto-response is worse than a delayed human response |
| 6 | **Escalation paths** (auto → human) are essential — auto-resolution should never be a dead end |
| 7 | **Every routing path must be tested** — easy/auto, hard/human, and escalation each need test cases |
| 8 | **Track routing metrics** — accuracy, false positives (dangerous), false negatives (wasteful) |
| 9 | **Threshold tuning** is a business decision: balance human agent cost vs customer satisfaction risk |
| 10 | **State schemas make graphs debuggable** — you can inspect exactly what each node produced and why routing went a certain way |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*